# 4. LLM-агент: гипотезы о механизме

**Цели:** получить проверяемые гипотезы, а не свободное описание графика. Агенту передаются данные и строгий контракт: утверждение, тест, критерий опровержения и ограничение.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/fractal_extreme_series.csv").exists())
DATA = ROOT / "data/fractal_extreme_series.csv"
SEED = 42
rng = np.random.default_rng(SEED)
df = pd.read_csv(DATA)
x = df["value"].to_numpy()
assert np.isfinite(x).all() and x.ndim == 1
# Graph-EVT-agent is the analysis library; no model code is copied into this repository.
from graph_evt_agent import ResearchAgent, ResearchTask, LLMConfig

llm = LLMConfig.from_env()  # provider/model/key are selected with environment variables
agent = ResearchAgent(llm=llm, seed=SEED)


In [ ]:
task=ResearchTask(
 question="Связаны ли экстремумы с локальной волатильностью и масштабной зависимостью?", data=x,
 methods=["rolling_volatility","extremogram","block_bootstrap"],
 output_schema={"hypotheses":"list","tests":"table","limitations":"list"},
 constraints={"seed":SEED,"alpha":0.05,"max_hypotheses":3},
)
hypotheses=agent.run(task)
hypotheses.tables["tests"]


In [ ]:
hypotheses.save(ROOT/'results/04_hypotheses')
hypotheses.summary


Формулировки агента считаются предложениями для проверки. Статистическая значимость не устанавливает причинность.

**Самопроверка:** что делает гипотезу опровержимой? Почему нужен block bootstrap? Где возникает множественная проверка?